In [1]:
import pandas as pd

#### prep bus and mrt data for the datasets

In [2]:
bus_data = pd.read_csv('../../data/A0008D - v_q_bus_stop (full).csv')
bus_location_data = pd.read_csv('../../data/correct_bus_location.csv')
bus_additional_data = pd.read_csv('../../data/A0008D - v_q_vls_marker (full).csv')

In [3]:
bus_consolidated = bus_data.merge(
   bus_location_data,
   how='outer',
   left_on='BUS_STOP_CD',
   right_on='BusStopCode'
)


bus_needed = bus_consolidated[[
   "BUS_STOP_CD",
   "BUS_STOP_NAM",
   "MRK_ID_NUM",
   "BusStopCode",
   "Description",
   "Latitude",
   "Longitude"
]].copy()


bus_needed = bus_needed.rename(columns={
   "BUS_STOP_NAM": "STATION_NAME",
   "Latitude": "LATITUDE",
   "Longitude": "LONGITUDE"
})


bus_needed = bus_needed.rename(columns={
   "BUS_STOP_NAM": "STATION_NAME",
   "Description": "LTA_DESCRIPTION",
   "Latitude": "LATITUDE",
   "Longitude": "LONGITUDE"
})
bus_needed["FINAL_STOP_CODE"] = bus_needed["BUS_STOP_CD"].fillna(bus_needed["BusStopCode"])
bus_needed["STATION_NAME"] = bus_needed["STATION_NAME"].fillna(bus_needed["LTA_DESCRIPTION"])
bus_needed['Travel_Type'] = 'BUS'


bus_needed_final = bus_needed[[
   "FINAL_STOP_CODE",
   "STATION_NAME",
   "MRK_ID_NUM",
   "LATITUDE",
   "LONGITUDE",
   "Travel_Type"
]].copy()


bus_additional_data["LATITUDE"] = bus_additional_data["LATITUDE_VAL"] / 3600000
bus_additional_data["LONGITUDE"] = bus_additional_data["LONGITUDE_VAL"] / 3600000


bus_additional_data_clean = bus_additional_data[[
   "MRK_ID_NUM",
   "LATITUDE",
   "LONGITUDE"
]].copy()


bus_needed_final = bus_needed_final.merge(
   bus_additional_data_clean,
   on="MRK_ID_NUM",
   how="left",
   suffixes=("", "_additional")
)




bus_needed_final["LATITUDE"] = bus_needed_final["LATITUDE"].fillna(
   bus_needed_final["LATITUDE_additional"]
)


bus_needed_final["LONGITUDE"] = bus_needed_final["LONGITUDE"].fillna(
   bus_needed_final["LONGITUDE_additional"]
)


bus_needed_final = bus_needed_final.drop(columns=[
   "LATITUDE_additional",
   "LONGITUDE_additional"
])


print("Remaining missing coords:",
     bus_needed_final["LATITUDE"].isna().sum())

Remaining missing coords: 0


In [4]:
train_data = pd.read_csv('../../data/mrt_station_coordinates.csv')
train_needed = train_data[['NODE_NAME', 'NODE_MRK_ID', 'LATITUDE', 'LONGITUDE']]
train_needed = train_needed.rename(columns={
   'NODE_NAME': 'STATION_NAME',
   'NODE_MRK_ID': 'MRK_ID_NUM'
})
train_needed['Travel_Type'] = 'TRAIN'
train_needed.head(5)


,STATION_NAME,MRK_ID_NUM,LATITUDE,LONGITUDE,Travel_Type
0,Yio Chu Kang,1,1.381756,103.844947,TRAIN
1,Ang Mo Kio,2,1.369933,103.849558,TRAIN
2,Bishan NSEW,3,1.350839,103.848144,TRAIN
3,Braddell,4,1.340469,103.846799,TRAIN
4,Toa Payoh,5,1.332629,103.847502,TRAIN


In [5]:
bus_needed_final = bus_needed_final[[
   "STATION_NAME",
   "MRK_ID_NUM",
   "LATITUDE",
   "LONGITUDE",
   "Travel_Type"
]]


consolidated_stations = pd.concat(
   [bus_needed_final, train_needed],
   axis=0,
   ignore_index=True
)


consolidated_stations.tail(5)

,STATION_NAME,MRK_ID_NUM,LATITUDE,LONGITUDE,Travel_Type
6271,Marine Parade,427.0,1.302865,103.905508,TRAIN
6272,Marine Terrace,428.0,1.306786,103.915316,TRAIN
6273,Siglap,429.0,1.309877,103.929879,TRAIN
6274,Bayshore,430.0,1.312841,103.941597,TRAIN
6275,Hume,336.0,1.354511,103.769104,TRAIN


#### for pt_ride

In [6]:
df = pd.read_csv('../../data/A0008D - v_pt_ride_all (full).csv')

In [7]:
df['BIZ_DT'] = pd.to_datetime(df['BIZ_DT'])
df['ENTRY_DT'] = pd.to_datetime(df['ENTRY_DT'])
df['EXIT_DT'] = pd.to_datetime(df['EXIT_DT'])
df['ENTRY_TM'] = pd.to_datetime(
   df['ENTRY_DT'].astype(str) + ' ' + df['ENTRY_TM'],
   format='%Y-%m-%d %H:%M:%S.%f'
)
df['EXIT_TM'] = pd.to_datetime(
   df['EXIT_DT'].astype(str) + ' ' + df['EXIT_TM'],
   format='%Y-%m-%d %H:%M:%S.%f'
)


In [8]:
df.dtypes

BIZ_DT                datetime64[us]
BUS_SVC_NUM                  float64
CRD_NUM                        int64
DEST_LOC_ID_NUM                int64
ENTRY_DT              datetime64[us]
ENTRY_TM              datetime64[us]
EXIT_DT               datetime64[us]
EXIT_TM               datetime64[us]
JRNY_ID_NUM                    int64
ORIG_LOC_ID_NUM                int64
PATRON_CATG_ID_NUM             int64
PAY_CD                         int64
RIDE_DISC_AMT                float64
RIDE_DIST_KM_CNT             float64
RIDE_FARE_AMT                float64
RIDE_ID_NUM                    int64
RIDE_MIN_CNT                 float64
RIDE_ST_CD                     int64
SVC_GRADE_ID_NUM               int64
TKT_TYP_CD                     int64
TRNSPT_MODE_CD                 int64
dtype: object

In [9]:
# df1 = df[df['ENTRY_DT'] == '2025-02-11'].copy()
# df1.columns
df2 = df[['BUS_SVC_NUM', 'CRD_NUM', 'DEST_LOC_ID_NUM', 'ENTRY_DT',
      'ENTRY_TM', 'EXIT_DT', 'EXIT_TM', 'JRNY_ID_NUM', 'ORIG_LOC_ID_NUM', 'RIDE_DISC_AMT', 'RIDE_DIST_KM_CNT',
      'RIDE_FARE_AMT', 'RIDE_ID_NUM', 'RIDE_MIN_CNT','PATRON_CATG_ID_NUM','TRNSPT_MODE_CD']]
df2.head()

,BUS_SVC_NUM,CRD_NUM,DEST_LOC_ID_NUM,ENTRY_DT,ENTRY_TM,EXIT_DT,EXIT_TM,JRNY_ID_NUM,ORIG_LOC_ID_NUM,RIDE_DISC_AMT,RIDE_DIST_KM_CNT,RIDE_FARE_AMT,RIDE_ID_NUM,RIDE_MIN_CNT,PATRON_CATG_ID_NUM,TRNSPT_MODE_CD
0,NaN,9200002624150,402,2025-02-22,2025-02-22 12:37:05,2025-02-22,2025-02-22 12:37:29,110780691633,403,0.0,1.4,0.69,110926783334,0.400,4,2
1,187.0,240000373709,2753,2025-02-22,2025-02-22 10:05:07,2025-02-22,2025-02-22 10:09:32,110780657323,2767,0.0,1.1,0.52,110926750879,4.417,3,1
2,168.0,9200002624150,4344,2025-02-22,2025-02-22 12:46:47,2025-02-22,2025-02-22 13:15:51,110780691633,5060,0.0,13.3,0.34,110926783335,29.067,4,1
3,NaN,240000373709,17,2025-02-22,2025-02-22 10:40:23,2025-02-22,2025-02-22 10:41:11,110780657323,28,0.0,12.4,0.22,110926750880,0.800,3,2
4,NaN,9190001883020,41,2025-02-22,2025-02-22 10:30:17,2025-02-22,2025-02-22 10:39:53,110780692295,42,0.0,2.4,0.00,110926789676,9.600,3,2


In [10]:
df2 = df2.merge(consolidated_stations, how = 'left', left_on= 'DEST_LOC_ID_NUM',
               right_on= 'MRK_ID_NUM')
df2.head(5)

,BUS_SVC_NUM,CRD_NUM,DEST_LOC_ID_NUM,ENTRY_DT,ENTRY_TM,EXIT_DT,EXIT_TM,JRNY_ID_NUM,ORIG_LOC_ID_NUM,RIDE_DISC_AMT,...,RIDE_FARE_AMT,RIDE_ID_NUM,RIDE_MIN_CNT,PATRON_CATG_ID_NUM,TRNSPT_MODE_CD,STATION_NAME,MRK_ID_NUM,LATITUDE,LONGITUDE,Travel_Type
0,NaN,9200002624150,402,2025-02-22,2025-02-22 12:37:05,2025-02-22,2025-02-22 12:37:29,110780691633,403,0.0,...,0.69,110926783334,0.400,4,2,Woodlands TEL,402.0,1.436058,103.787939,TRAIN
1,187.0,240000373709,2753,2025-02-22,2025-02-22 10:05:07,2025-02-22,2025-02-22 10:09:32,110780657323,2767,0.0,...,0.52,110926750879,4.417,3,1,Opp Blk 628,2753.0,1.351765,103.750199,BUS
2,168.0,9200002624150,4344,2025-02-22,2025-02-22 12:46:47,2025-02-22,2025-02-22 13:15:51,110780691633,5060,0.0,...,0.34,110926783335,29.067,4,1,Bef Seletar Camp G,4344.0,1.402222,103.871388,BUS
3,NaN,240000373709,17,2025-02-22,2025-02-22 10:40:23,2025-02-22,2025-02-22 10:41:11,110780657323,28,0.0,...,0.22,110926750880,0.800,3,2,Redhill,17.0,1.289635,103.816741,TRAIN
4,NaN,9190001883020,41,2025-02-22,2025-02-22 10:30:17,2025-02-22,2025-02-22 10:39:53,110780692295,42,0.0,...,0.00,110926789676,9.600,3,2,Tampines NSEW,41.0,1.353302,103.945145,TRAIN


In [11]:
df2.rename(columns={
   'STATION_NAME': 'DEST_STATION_NAME',
   'MRK_ID_NUM': 'DEST_MRK_ID_NUM',
   'LATITUDE': 'DEST_LATITUDE',
   'LONGITUDE': 'DEST_LONGITUDE',
   'Travel_Type': 'DEST_Travel_Type'
}, inplace=True)

In [12]:
df2 = df2.merge(consolidated_stations, how = 'left', left_on= 'ORIG_LOC_ID_NUM',
               right_on= 'MRK_ID_NUM')

In [13]:
df2.rename(columns={
   'STATION_NAME': 'ORIG_STATION_NAME',
   'MRK_ID_NUM': 'ORIG_MRK_ID_NUM',
   'LATITUDE': 'ORIG_LATITUDE',
   'LONGITUDE': 'ORIG_LONGITUDE',
   'Travel_Type': 'ORIG_Travel_Type'
}, inplace=True)
df2.head(5)

,BUS_SVC_NUM,CRD_NUM,DEST_LOC_ID_NUM,ENTRY_DT,ENTRY_TM,EXIT_DT,EXIT_TM,JRNY_ID_NUM,ORIG_LOC_ID_NUM,RIDE_DISC_AMT,...,DEST_STATION_NAME,DEST_MRK_ID_NUM,DEST_LATITUDE,DEST_LONGITUDE,DEST_Travel_Type,ORIG_STATION_NAME,ORIG_MRK_ID_NUM,ORIG_LATITUDE,ORIG_LONGITUDE,ORIG_Travel_Type
0,NaN,9200002624150,402,2025-02-22,2025-02-22 12:37:05,2025-02-22,2025-02-22 12:37:29,110780691633,403,0.0,...,Woodlands TEL,402.0,1.436058,103.787939,TRAIN,Woodlands South,403.0,1.427396,103.793264,TRAIN
1,187.0,240000373709,2753,2025-02-22,2025-02-22 10:05:07,2025-02-22,2025-02-22 10:09:32,110780657323,2767,0.0,...,Opp Blk 628,2753.0,1.351765,103.750199,BUS,Blk 315,2767.0,1.359929,103.746379,BUS
2,168.0,9200002624150,4344,2025-02-22,2025-02-22 12:46:47,2025-02-22,2025-02-22 13:15:51,110780691633,5060,0.0,...,Bef Seletar Camp G,4344.0,1.402222,103.871388,BUS,Woodlands Int B10,5060.0,1.436946,103.785936,BUS
3,NaN,240000373709,17,2025-02-22,2025-02-22 10:40:23,2025-02-22,2025-02-22 10:41:11,110780657323,28,0.0,...,Redhill,17.0,1.289635,103.816741,TRAIN,Bukit Batok,28.0,1.349033,103.749567,TRAIN
4,NaN,9190001883020,41,2025-02-22,2025-02-22 10:30:17,2025-02-22,2025-02-22 10:39:53,110780692295,42,0.0,...,Tampines NSEW,41.0,1.353302,103.945145,TRAIN,Pasir Ris,42.0,1.373043,103.949285,TRAIN


In [14]:
# Convert to pickle to reference the dataframe in other files
df2.to_pickle('df2.pkl')
print('Saved df2.pkl')

Saved df2.pkl


In [15]:
# To load the dataframe in another file later
# df2 = pd.read_pickle('df2.pkl')

#### for pt_jrny

In [16]:
df4 = pd.read_csv('../../data/A0008D - v_pt_jrny_all (full).csv')

In [17]:
df4['BIZ_DT'] = pd.to_datetime(df4['BIZ_DT'])
df4['JRNY_START_DT'] = pd.to_datetime(df4['JRNY_START_DT'])
df4['JRNY_END_DT'] = pd.to_datetime(df4['JRNY_END_DT'])
df4['JRNY_START_TM'] = pd.to_datetime(
   df4['JRNY_START_DT'].astype(str) + ' ' + df4['JRNY_START_TM'],
   format='%Y-%m-%d %H:%M:%S.%f'
)
df4['JRNY_END_TM'] = pd.to_datetime(
   df4['JRNY_END_DT'].astype(str) + ' ' + df4['JRNY_END_TM'],
   format='%Y-%m-%d %H:%M:%S.%f'
)

In [18]:
df4.dtypes

BIZ_DT                datetime64[us]
CRD_NUM                        int64
JRNY_DEST_ID_NUM             float64
JRNY_DIST_KM_CNT             float64
JRNY_END_DT           datetime64[us]
JRNY_END_TM           datetime64[us]
JRNY_FARE_AMT                float64
JRNY_ID_NUM                    int64
JRNY_ORIG_ID_NUM             float64
JRNY_ST_CD                     int64
JRNY_START_DT         datetime64[us]
JRNY_START_TM         datetime64[us]
JRNY_TM_MIN_CNT              float64
PATRON_CATG_ID_NUM             int64
PAY_CD                         int64
TKT_TYP_CD                     int64
TRNSPT_MODE_CD                 int64
dtype: object

In [19]:
# df4 = d4[df4['JRNY_START_DT'] == '2025-02-11'].copy()
# df4.columns
df5 = df4[['CRD_NUM', 'JRNY_DEST_ID_NUM', 'JRNY_START_DT',
      'JRNY_START_TM', 'JRNY_END_DT', 'JRNY_END_TM', 'JRNY_ORIG_ID_NUM', 'JRNY_DIST_KM_CNT',
      'JRNY_FARE_AMT', 'JRNY_ID_NUM', 'JRNY_TM_MIN_CNT','PATRON_CATG_ID_NUM','TRNSPT_MODE_CD']]
df5.head()

,CRD_NUM,JRNY_DEST_ID_NUM,JRNY_START_DT,JRNY_START_TM,JRNY_END_DT,JRNY_END_TM,JRNY_ORIG_ID_NUM,JRNY_DIST_KM_CNT,JRNY_FARE_AMT,JRNY_ID_NUM,JRNY_TM_MIN_CNT,PATRON_CATG_ID_NUM,TRNSPT_MODE_CD
0,1700172626889,106.0,2025-02-16,2025-02-16 10:27:50,2025-02-16,2025-02-16 10:45:37,112.0,6.6,1.59,110731641253,17.783,1,2
1,9180002129550,4575.0,2025-02-16,2025-02-16 15:00:45,2025-02-16,2025-02-16 15:09:21,6838.0,1.4,0.69,110731641254,8.600,4,1
2,2220003302277,66.0,2025-02-16,2025-02-16 20:38:41,2025-02-16,2025-02-16 20:52:21,24.0,5.6,1.50,110731641255,13.667,1,2
3,9210002187503,6474.0,2025-02-16,2025-02-16 19:43:43,2025-02-16,2025-02-16 21:01:30,207.0,31.7,1.03,110731641256,77.783,4,3
4,230005862960,4152.0,2025-02-16,2025-02-16 21:06:31,2025-02-16,2025-02-16 21:22:54,4193.0,4.2,0.76,110731641257,16.383,4,1


In [20]:
df5 = df5.merge(consolidated_stations, how = 'left', left_on= 'JRNY_DEST_ID_NUM',
               right_on= 'MRK_ID_NUM')
df5.head(5)

,CRD_NUM,JRNY_DEST_ID_NUM,JRNY_START_DT,JRNY_START_TM,JRNY_END_DT,JRNY_END_TM,JRNY_ORIG_ID_NUM,JRNY_DIST_KM_CNT,JRNY_FARE_AMT,JRNY_ID_NUM,JRNY_TM_MIN_CNT,PATRON_CATG_ID_NUM,TRNSPT_MODE_CD,STATION_NAME,MRK_ID_NUM,LATITUDE,LONGITUDE,Travel_Type
0,1700172626889,106.0,2025-02-16,2025-02-16 10:27:50,2025-02-16,2025-02-16 10:45:37,112.0,6.6,1.59,110731641253,17.783,1,2,Serangoon NEL,106.0,1.349708,103.873575,TRAIN
1,9180002129550,4575.0,2025-02-16,2025-02-16 15:00:45,2025-02-16,2025-02-16 15:09:21,6838.0,1.4,0.69,110731641254,8.600,4,1,Blk 496F,4575.0,1.360256,103.950223,BUS
2,2220003302277,66.0,2025-02-16,2025-02-16 20:38:41,2025-02-16,2025-02-16 20:52:21,24.0,5.6,1.50,110731641255,13.667,1,2,Pioneer,66.0,1.337587,103.697322,TRAIN
3,9210002187503,6474.0,2025-02-16,2025-02-16 19:43:43,2025-02-16,2025-02-16 21:01:30,207.0,31.7,1.03,110731641256,77.783,4,3,Alight Johor Bahru CP,6474.0,1.464917,103.765477,BUS
4,230005862960,4152.0,2025-02-16,2025-02-16 21:06:31,2025-02-16,2025-02-16 21:22:54,4193.0,4.2,0.76,110731641257,16.383,4,1,Hougang 1,4152.0,1.375130,103.879052,BUS


In [21]:
df5.rename(columns={
   'STATION_NAME': 'DEST_STATION_NAME',
   'MRK_ID_NUM': 'DEST_MRK_ID_NUM',
   'LATITUDE': 'DEST_LATITUDE',
   'LONGITUDE': 'DEST_LONGITUDE',
   'Travel_Type': 'DEST_Travel_Type'
}, inplace=True)

In [22]:
df5 = df5.merge(consolidated_stations, how = 'left', left_on= 'JRNY_ORIG_ID_NUM',
               right_on= 'MRK_ID_NUM')

In [23]:
df5.rename(columns={
   'STATION_NAME': 'ORIG_STATION_NAME',
   'MRK_ID_NUM': 'ORIG_MRK_ID_NUM',
   'LATITUDE': 'ORIG_LATITUDE',
   'LONGITUDE': 'ORIG_LONGITUDE',
   'Travel_Type': 'ORIG_Travel_Type'
}, inplace=True)
df5.head(5)

,CRD_NUM,JRNY_DEST_ID_NUM,JRNY_START_DT,JRNY_START_TM,JRNY_END_DT,JRNY_END_TM,JRNY_ORIG_ID_NUM,JRNY_DIST_KM_CNT,JRNY_FARE_AMT,JRNY_ID_NUM,...,DEST_STATION_NAME,DEST_MRK_ID_NUM,DEST_LATITUDE,DEST_LONGITUDE,DEST_Travel_Type,ORIG_STATION_NAME,ORIG_MRK_ID_NUM,ORIG_LATITUDE,ORIG_LONGITUDE,ORIG_Travel_Type
0,1700172626889,106.0,2025-02-16,2025-02-16 10:27:50,2025-02-16,2025-02-16 10:45:37,112.0,6.6,1.59,110731641253,...,Serangoon NEL,106.0,1.349708,103.873575,TRAIN,Dhoby Ghaut NEL,112.0,1.299705,103.845488,TRAIN
1,9180002129550,4575.0,2025-02-16,2025-02-16 15:00:45,2025-02-16,2025-02-16 15:09:21,6838.0,1.4,0.69,110731641254,...,Blk 496F,4575.0,1.360256,103.950223,BUS,Tampines Int Alighting 1,6838.0,1.354076,103.943391,BUS
2,2220003302277,66.0,2025-02-16,2025-02-16 20:38:41,2025-02-16,2025-02-16 20:52:21,24.0,5.6,1.50,110731641255,...,Pioneer,66.0,1.337587,103.697322,TRAIN,Jurong East,24.0,1.333153,103.742286,TRAIN
3,9210002187503,6474.0,2025-02-16,2025-02-16 19:43:43,2025-02-16,2025-02-16 21:01:30,207.0,31.7,1.03,110731641256,...,Alight Johor Bahru CP,6474.0,1.464917,103.765477,BUS,MacPherson CCL,207.0,1.326078,103.890391,TRAIN
4,230005862960,4152.0,2025-02-16,2025-02-16 21:06:31,2025-02-16,2025-02-16 21:22:54,4193.0,4.2,0.76,110731641257,...,Hougang 1,4152.0,1.375130,103.879052,BUS,S'Goon Int Boarding Berth 2,4193.0,1.350466,103.871690,BUS


In [24]:
#df5.to_pickle('df5.pkl')
#print('Saved df5.pkl')

In [25]:
#df2 = df2[df2['ENTRY_DT'] == '2025-02-11'].copy()
df2 = df2[df2["ENTRY_DT"].dt.date == pd.Timestamp("2025-02-11").date()].copy()
df3 = df2.sort_values(["CRD_NUM", "ENTRY_DT", "ENTRY_TM"], ascending= [True, True, True]).reset_index(drop= True)

In [26]:
# Onemap API does not allow dates more than a year
df3["ENTRY_DT"] = pd.Timestamp("10-04-2025")
df3["EXIT_DT"] = pd.Timestamp("10-04-2025")




df3["ENTRY_TM"] = df3["ENTRY_TM"].dt.strftime("%H:%M:%S")
df3["EXIT_TM"] = df3["EXIT_TM"].dt.strftime("%H:%M:%S")
df3["ENTRY_DT"] = df3["ENTRY_DT"].dt.strftime("%m-%d-%Y")
df3["EXIT_DT"] = df3["EXIT_DT"].dt.strftime("%m-%d-%Y")

In [27]:
df3["next_orig_lat"] = df3.groupby("CRD_NUM")["ORIG_LATITUDE"].shift(-1)
df3["next_orig_lon"] = df3.groupby("CRD_NUM")["ORIG_LONGITUDE"].shift(-1)
df3["next_orig_station"] = df3.groupby("CRD_NUM")["ORIG_STATION_NAME"].shift(-1)

In [28]:
walk_cache = pd.read_csv('../../data/walk_distance_cache.csv')

In [29]:
df3["pair_key"] = (
  df3["DEST_LATITUDE"].astype(str) + "_" +
  df3["DEST_LONGITUDE"].astype(str) + "_" +
  df3["next_orig_lat"].astype(str) + "_" +
  df3["next_orig_lon"].astype(str)
)




pairs = df3[
   [
       "pair_key",
       "DEST_STATION_NAME",
       "next_orig_station",
       "DEST_LATITUDE",
       "DEST_LONGITUDE",
       "next_orig_lat",
       "next_orig_lon"
   ]
].dropna(subset=[
   "DEST_LATITUDE",
   "DEST_LONGITUDE",
   "next_orig_lat",
   "next_orig_lon"
]).drop_duplicates("pair_key").copy()


In [30]:
df3 = (
   df3.drop(columns=["pair_key"], errors="ignore")
   .merge(
       walk_cache.drop(columns=["pair_key"], errors="ignore")[[
           "DEST_LATITUDE", "DEST_LONGITUDE", "next_orig_lat", "next_orig_lon", "walk_distance"
       ]].drop_duplicates(
           subset=["DEST_LATITUDE", "DEST_LONGITUDE", "next_orig_lat", "next_orig_lon"]
       ),
       on=["DEST_LATITUDE", "DEST_LONGITUDE", "next_orig_lat", "next_orig_lon"],
       how="left"
   )
)


In [31]:
# restore the actual filtered business date
real_date = pd.Timestamp("2025-02-11")


df3["ENTRY_DT"] = real_date
df3["EXIT_DT"] = real_date


# if ENTRY_TM / EXIT_TM are currently strings like '00:47:26',
# convert them back into full datetimes
df3["ENTRY_TM"] = pd.to_datetime(
   df3["ENTRY_DT"].astype(str) + " " + df3["ENTRY_TM"].astype(str),
   format="%Y-%m-%d %H:%M:%S",
   errors="coerce"
)


df3["EXIT_TM"] = pd.to_datetime(
   df3["EXIT_DT"].astype(str) + " " + df3["EXIT_TM"].astype(str),
   format="%Y-%m-%d %H:%M:%S",
   errors="coerce"
)


In [32]:
df3.head(20)

,BUS_SVC_NUM,CRD_NUM,DEST_LOC_ID_NUM,ENTRY_DT,ENTRY_TM,EXIT_DT,EXIT_TM,JRNY_ID_NUM,ORIG_LOC_ID_NUM,RIDE_DISC_AMT,...,DEST_Travel_Type,ORIG_STATION_NAME,ORIG_MRK_ID_NUM,ORIG_LATITUDE,ORIG_LONGITUDE,ORIG_Travel_Type,next_orig_lat,next_orig_lon,next_orig_station,walk_distance
0,NaN,100005879220,22,2025-02-11,2025-02-11 11:20:34,2025-02-11,2025-02-11 11:32:25,110710909992,44,0.00,...,TRAIN,Admiralty,44.0,1.440589,103.800990,TRAIN,1.429443,103.835005,Yishun,0.0
1,NaN,100005879220,44,2025-02-11,2025-02-11 22:32:59,2025-02-11,2025-02-11 22:45:03,110710523649,22,0.00,...,TRAIN,Yishun,22.0,1.429443,103.835005,TRAIN,NaN,NaN,NaN,NaN
2,45.0,100006223599,3968,2025-02-11,2025-02-11 06:56:13,2025-02-11,2025-02-11 07:07:33,110710081977,4009,0.00,...,BUS,Blk 172,4009.0,1.350814,103.889844,BUS,1.349708,103.873575,Serangoon NEL,195.0
3,NaN,100006223599,109,2025-02-11,2025-02-11 07:09:17,2025-02-11,2025-02-11 07:18:09,110710081977,106,0.16,...,TRAIN,Serangoon NEL,106.0,1.349708,103.873575,TRAIN,1.319336,103.861570,Boon Keng,0.0
4,NaN,100006223599,106,2025-02-11,2025-02-11 14:51:22,2025-02-11,2025-02-11 15:03:53,110710904744,109,0.00,...,TRAIN,Boon Keng,109.0,1.319336,103.861570,TRAIN,1.348979,103.872774,S'Goon Stn Exit B,139.0
5,45.0,100006223599,4008,2025-02-11,2025-02-11 15:10:02,2025-02-11,2025-02-11 15:21:55,110710226201,3967,0.00,...,BUS,S'Goon Stn Exit B,3967.0,1.348979,103.872774,BUS,NaN,NaN,NaN,NaN
6,382.0,130013244516,5960,2025-02-11,2025-02-11 10:07:29,2025-02-11,2025-02-11 10:11:38,110710535706,5953,0.00,...,BUS,Blk 322 CP,5953.0,1.410690,103.897592,BUS,1.408195,103.905669,One Punggol,123.0
7,382.0,130013244516,5954,2025-02-11,2025-02-11 12:15:49,2025-02-11,2025-02-11 12:21:50,110709913132,5959,0.00,...,BUS,One Punggol,5959.0,1.408195,103.905669,BUS,NaN,NaN,NaN,NaN
8,871.0,130013244638,2782,2025-02-11,2025-02-11 06:40:58,2025-02-11,2025-02-11 06:54:37,110711365561,6011,0.00,...,BUS,Tengah CC,6011.0,1.356473,103.734424,BUS,1.358612,103.751791,Bukit Gombak,248.0
9,NaN,130013244638,45,2025-02-11,2025-02-11 06:58:38,2025-02-11,2025-02-11 07:20:00,110711365561,29,0.50,...,TRAIN,Bukit Gombak,29.0,1.358612,103.751791,TRAIN,1.436946,103.785936,Woodlands Int B5,25.0


In [33]:
df3.to_pickle('df3.pkl')
print('Saved df3.pkl')

Saved df3.pkl


In [34]:
#df5.to_pickle('df5.pkl')
#print('Saved df5.pkl')